# Fate RAG — Exploration & Experimentation Notebook

This notebook walks through the full RAG pipeline:
1. **Load** sample / scraped data
2. **Chunk** documents and visualize token distributions
3. **Embed** a sample with Bedrock Titan
4. **Query** OpenSearch with a test question
5. **Generate** a response using Claude

Make sure you have started the local OpenSearch instance:
```bash
docker-compose up -d
```
and installed dependencies:
```bash
pip install -r requirements.txt
```

In [ ]:
import sys, os
# Add project root to path so we can import local modules
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment loaded.')

## 1. Load Sample Data

In [ ]:
import json
from pathlib import Path

sample_dir = Path('../sample_data')
docs = []
for f in sample_dir.glob('*.json'):
    with open(f) as fp:
        data = json.load(fp)
        if isinstance(data, list):
            docs.extend(data)
        else:
            docs.append(data)

print(f'Loaded {len(docs)} documents from sample_data/')
for doc in docs:
    print(f"  [{doc.get('category','?')}] {doc.get('title','?')} — {len(doc.get('content',''))} chars")

## 2. Chunking & Token Distribution

In [ ]:
import tiktoken
import matplotlib.pyplot as plt

from data_pipeline.chunker import chunk_document, _get_encoder

encoder = _get_encoder()
all_chunks = []
for doc in docs:
    chunks = chunk_document(doc, encoder)
    all_chunks.extend(chunks)

print(f'Total chunks: {len(all_chunks)}')
token_counts = [c['token_count'] for c in all_chunks]

plt.figure(figsize=(8, 4))
plt.hist(token_counts, bins=20, color='#7c5cbf', edgecolor='black', alpha=0.85)
plt.title('Chunk Token Distribution')
plt.xlabel('Tokens per chunk')
plt.ylabel('Frequency')
plt.axvline(x=500, color='#c8a45e', linestyle='--', label='Target (500 tokens)')
plt.legend()
plt.tight_layout()
plt.show()

import statistics
print(f'Mean: {statistics.mean(token_counts):.1f}')
print(f'Median: {statistics.median(token_counts):.1f}')
print(f'Max: {max(token_counts)}')
print(f'Min: {min(token_counts)}')

In [ ]:
# Inspect a sample chunk
import pprint
sample_chunk = all_chunks[0]
pprint.pprint({k: v for k, v in sample_chunk.items() if k != 'text'})
print('\n--- TEXT PREVIEW ---')
print(sample_chunk['text'][:500])

## 3. Embed a Sample with Bedrock Titan

Requires valid AWS credentials and Bedrock access.

In [ ]:
from data_pipeline.embedder import _get_bedrock_client, embed_text

bedrock_client = _get_bedrock_client()
test_text = all_chunks[0]['text'][:512]
vector = embed_text(test_text, bedrock_client)

print(f'Embedding dims: {len(vector)}')
print(f'First 5 values: {vector[:5]}')

## 4. Upload to Local OpenSearch

Make sure `docker-compose up -d` is running.

In [ ]:
from data_pipeline.embedder import _get_opensearch_client, ensure_index, embed_and_upload

os_client = _get_opensearch_client()
print('OpenSearch info:', os_client.info()['version']['number'])

n = embed_and_upload(all_chunks, os_client, bedrock_client)
print(f'Uploaded {n} chunks to OpenSearch.')

## 5. Retrieval Test

In [ ]:
from backend.retriever import FateRetriever

retriever = FateRetriever(os_client=os_client, bedrock_client=bedrock_client)

queries = [
    'What is the Noble Phantasm of Saber?',
    'How does the Holy Grail War work?',
    'Who summoned Gilgamesh in Fate/Zero?',
]

for q in queries:
    print(f'\n\033[1mQ: {q}\033[0m')
    results = retriever.retrieve(q, top_k=3)
    for i, r in enumerate(results, 1):
        print(f'  [{i}] {r["title"]} (score={r["score"]:.3f}, cat={r["category"]})')
        print(f'      {r["text"][:120]}...')

## 6. End-to-End Generation

In [ ]:
import boto3, json, os
from backend.prompt import SYSTEM_PROMPT, build_messages

MODEL_ID = os.getenv('BEDROCK_MODEL_ID', 'claude-sonnet-4-20250514')
bedrock_gen = boto3.client('bedrock-runtime', region_name=os.getenv('BEDROCK_REGION', 'us-east-1'))

question = "What is Unlimited Blade Works and who uses it?"
retrieved = retriever.retrieve(question, top_k=3)
messages = build_messages(question=question, retrieved_docs=retrieved)

response = bedrock_gen.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps({
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': 512,
        'system': SYSTEM_PROMPT,
        'messages': messages,
    }),
    contentType='application/json',
    accept='application/json',
)

result = json.loads(response['body'].read())
print(result['content'][0]['text'])

## 7. Chunking Strategy Comparison

Compare different chunk sizes to see how they affect retrieval quality.

In [ ]:
from data_pipeline.chunker import _split_into_chunks

test_doc = docs[0]['content'] if docs else 'No documents loaded.'
configs = [(256, 25), (500, 50), (1000, 100)]

print(f'Document length: {len(encoder.encode(test_doc))} tokens\n')
for size, overlap in configs:
    chunks = _split_into_chunks(test_doc, encoder, chunk_size=size, overlap=overlap)
    token_counts = [len(encoder.encode(c)) for c in chunks]
    print(f'chunk_size={size}, overlap={overlap}: {len(chunks)} chunks '
          f'(avg {sum(token_counts)//len(token_counts)} tokens)')